# Proof of Concept: Rubric-gebaseerde Formatieve Feedback met een Lokaal Gehost LLM

**Hoofdvraag:** Hoe kan een lokaal gehoste LLM worden ingezet voor het genereren van rubric-gebaseerde formatieve feedback op opleveringen van studenten binnen de opleiding Informatica van Hogeschool Leiden?

**Student:** Fariha Ahmad, s1140710 
**Opleiding:** HBO-ICT, Hogeschool Leiden  

---

## Setup & Configuratie

In deze sectie installeren we de benodigde pakketten, laden we de configuratie en testen we de verbinding met Ollama.

In [3]:
# Cel 0.1 — Pakketten installeren
# requests: HTTP-verzoeken naar Ollama (localhost:11434)
# pdfplumber: tekst extraheren uit PDF-bestanden
# spacy: Named Entity Recognition voor automatische anonimisering

%pip install requests pdfplumber spacy --quiet

# Download het Nederlandse SpaCy-taalmodel voor NER
!python -m spacy download nl_core_news_lg --quiet

print("Alle pakketten succesvol geïnstalleerd.")

Note: you may need to restart the kernel to use updated packages.
✔ Download and installation successful
You can now load the package via spacy.load('nl_core_news_lg')
Alle pakketten succesvol geïnstalleerd.


In [4]:
# Cel 0.2 — Imports en configuratie

import requests
import pdfplumber
import spacy
import json
import os
import re
from datetime import datetime

# === CONFIGURATIE ===
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen2.5:7b"

# Mappenstructuur
DATA_DIR = "data"              # Hier komen de PDF-bestanden
OUTPUT_DIR = "output"          # Hier komt de gegenereerde feedback
LOG_DIR = "logs"               # Hier komen de logbestanden

for directory in [DATA_DIR, OUTPUT_DIR, LOG_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f"Configuratie geladen:")
print(f"  Ollama URL: {OLLAMA_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  Mappen aangemaakt: {DATA_DIR}/, {OUTPUT_DIR}/, {LOG_DIR}/")

Configuratie geladen:
  Ollama URL: http://localhost:11434
  Model: qwen2.5:7b
  Mappen aangemaakt: data/, output/, logs/


In [5]:
# Cel 0.3 — Test of Ollama draait en het model beschikbaar is

def test_ollama_verbinding():
    """Test of Ollama bereikbaar is op localhost."""
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
        if response.status_code == 200:
            modellen = response.json().get("models", [])
            model_namen = [m["name"] for m in modellen]
            print(f"Ollama is bereikbaar!")
            print(f"Beschikbare modellen: {model_namen}")
            
            # Check of ons gewenste model ertussen staat
            if any(MODEL_NAME in naam for naam in model_namen):
                print(f"\n{MODEL_NAME} is beschikbaar.")
            else:
                print(f"\nLET OP: {MODEL_NAME} niet gevonden. Beschikbaar: {model_namen}")
                print(f"Draai in je terminal: ollama pull {MODEL_NAME}")
            return True
        else:
            print(f"Ollama gaf statuscode {response.status_code}")
            return False
    except requests.ConnectionError:
        print("FOUT: Kan geen verbinding maken met Ollama.")
        print("Controleer of Ollama draait via Pinokio op localhost:11434")
        return False

ollama_ok = test_ollama_verbinding()

Ollama is bereikbaar!
Beschikbare modellen: ['qwen2.5:7b', 'llama3.1:latest']

qwen2.5:7b is beschikbaar.


In [6]:
# Cel 0.4 — Stuur een testprompt naar het model

def stuur_testprompt():
    """Stuurt een simpele testprompt naar Ollama om te bevestigen dat het model werkt."""
    payload = {
        "model": MODEL_NAME,
        "prompt": "Antwoord exact met: Verbinding geslaagd.",
        "stream": False,
        "options": {
            "temperature": 0.1
        }
    }
    
    try:
        print(f"Testprompt versturen naar {MODEL_NAME}...")
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=300
        )
        
        if response.status_code == 200:
            resultaat = response.json()
            antwoord = resultaat.get("response", "")
            duur = resultaat.get("total_duration", 0) / 1e9  # nanoseconden naar seconden
            
            print(f"Antwoord van {MODEL_NAME}: {antwoord.strip()}")
            print(f"Responstijd: {duur:.2f} seconden")
            return True
        else:
            print(f"Fout: statuscode {response.status_code}")
            print(response.text)
            return False
    except requests.Timeout:
        print("FOUT: Timeout bij het aanroepen van Ollama.")
        print("Het model is mogelijk nog aan het laden. Probeer het opnieuw.")
        return False

if ollama_ok:
    model_ok = stuur_testprompt()
else:
    print("Sla deze stap over — Ollama is niet bereikbaar.")

Testprompt versturen naar qwen2.5:7b...
Antwoord van qwen2.5:7b: Verbinding geslaagd.
Responstijd: 48.58 seconden


---
## Sectie 1 — Data-inladen & Anonimisering

In deze sectie laden we de zes PvA-documenten in als PDF, extraheren we de tekst per pagina, en anonimiseren we automatisch namen en studentnummers met SpaCy NER. Afbeeldingen die niet verwerkt kunnen worden, worden gelogd.

### 1.1 — PDF-tekst extractie met afbeelding-detectie
Deze functie opent een PDF met pdfplumber, extraheert de tekst per pagina, en detecteert of er afbeeldingen op een pagina staan. Omdat de LLM geen afbeeldingen kan verwerken, loggen we precies welke pagina's afbeeldingen bevatten zodat transparant is wat er gemist is.

In [26]:
# Cel 1.1 — PDF-tekst extractie met afbeelding-detectie

def extract_pdf_tekst(pdf_pad):
    """
    Extraheert tekst uit een PDF-bestand.
    Logt per pagina of er afbeeldingen zijn die niet verwerkt kunnen worden.
    
    Returns:
        dict met:
        - "tekst": volledige geëxtraheerde tekst
        - "paginas": lijst met tekst per pagina
        - "afbeelding_log": lijst met pagina's die afbeeldingen bevatten
        - "metadata": bestandsnaam en aantal pagina's
    """
    resultaat = {
        "tekst": "",
        "paginas": [],
        "afbeelding_log": [],
        "metadata": {
            "bestandsnaam": os.path.basename(pdf_pad),
            "aantal_paginas": 0
        }
    }
    
    try:
        with pdfplumber.open(pdf_pad) as pdf:
            resultaat["metadata"]["aantal_paginas"] = len(pdf.pages)
            
            for i, pagina in enumerate(pdf.pages, start=1):
                # Tekst extraheren
                tekst = pagina.extract_text() or ""
                resultaat["paginas"].append(tekst)
                
                # Afbeeldingen detecteren
                afbeeldingen = pagina.images
                if afbeeldingen:
                    log_entry = {
                        "pagina": i,
                        "aantal_afbeeldingen": len(afbeeldingen),
                        "melding": f"Pagina {i} bevat {len(afbeeldingen)} afbeelding(en) die niet verwerkt kunnen worden door de LLM."
                    }
                    resultaat["afbeelding_log"].append(log_entry)
            
            resultaat["tekst"] = "\n\n".join(resultaat["paginas"])
    
    except FileNotFoundError:
        print(f"FOUT: Bestand niet gevonden: {pdf_pad}")
    except Exception as e:
        print(f"FOUT bij verwerken van {pdf_pad}: {e}")
    
    return resultaat

# Test met een voorbeeld
print("Functie extract_pdf_tekst() is geladen.")

Functie extract_pdf_tekst() is geladen.


### 1.2 — Automatische anonimisering met structurele patroonherkenning

We detecteren namen via drie regex-patronen die aansluiten op de werkelijke documentstructuur van de PvA's. Dit is betrouwbaarder dan SpaCy NER voor tabelregels en lijsten zonder zinscontext. SpaCy blijft optioneel als extra fallback voor namen in lopende tekst.

**Drie herkende patronen:**
- Groep B11-formaat: `Voornaam Achternaam - s1234567`
- Groep C8-formaat: `Voornaam Achternaam (1234567)`
- Groep A2-formaat: `Auteur(s): Naam1, Naam2, ...` (ook over meerdere regels)

**Extra:** namen die door PDF-extractie over twee regels zijn gebroken worden ook correct vervangen.

In [27]:
# Cel 1.2 — Automatische anonimisering met structurele patroonherkenning

import re
TUSSENVOEGSELS = r'van\s+den|van\s+der|van\s+de|van|de|den|der|te|ten'

# Optioneel: SpaCy als fallback voor namen in lopende tekst
try:
    import spacy
    nlp = spacy.load("nl_core_news_lg")
    SPACY_BESCHIKBAAR = True
    print("SpaCy geladen als aanvullende fallback.")
except Exception:
    SPACY_BESCHIKBAAR = False
    print("SpaCy niet beschikbaar. Structurele patronen worden gebruikt als primaire methode.")

def extract_namen_structureel(tekst):
    """
    Detecteert namen en studentnummers via structurele patronen
    die aansluiten op de documentformaten van de PvA's.

    Patroon 1 (B11): 'Voornaam Achternaam - s1234567'
    Patroon 2 (C8):  'Voornaam Achternaam (1234567)' of '(s1234567)'
    Patroon 3 (A2):  'Auteur(s): Naam1, Naam2, ...' (ook meerregelig)

    SpaCy NER wordt optioneel toegepast als extra fallback op lopende tekst.
    """
    namen = set()
    studentnrs = set()

    # === PATROON 1: "Voornaam Achternaam - s1234567" ===
    for m in re.finditer(
        r'([A-ZÁÉÍÓÚÀÈËÏÖÜ][a-záéíóúàèëïöü]+'
        r'(?:\s+(?:' + TUSSENVOEGSELS + r')(?=\s))?'
        r'\s+[A-ZÁÉÍÓÚÀÈËÏÖÜ][a-záéíóúàèëïöüA-ZÁÉÍÓÚÀÈËÏÖÜ\-]+)'
        r'\s*[-–]\s*[sS]?\d{6,8}',
        tekst
    ):
        namen.add(m.group(1))

    # === PATROON 2: "Voornaam Achternaam (1234567)" of "(s1234567)" ===
    for m in re.finditer(
        r'([A-ZÁÉÍÓÚÀÈËÏÖÜ][a-záéíóúàèëïöü]+'
        r'(?:\s+(?:' + TUSSENVOEGSELS + r')(?=\s))?'
        r'\s+[A-ZÁÉÍÓÚÀÈËÏÖÜ][a-záéíóúàèëïöüA-ZÁÉÍÓÚÀÈËÏÖÜ\-]+)'
        r'\s*\([sS]?\d{6,8}\)',
        tekst
    ):
        namen.add(m.group(1))

    # === PATROON 3: "Auteur(s): Naam1, Naam2, ..." ook over meerdere regels ===
    auteur_match = re.search(
        r'[Aa]uteur[s()\s]*:\s*(.+?)(?=\nStudentnummer|\nGroep|\nDatum|\Z)',
        tekst,
        re.DOTALL
    )
    if auteur_match:
        auteur_tekst = auteur_match.group(1).replace('\n', ' ')
        for k in auteur_tekst.split(','):
            k_schoon = k.strip()
            if re.match(r'^[A-ZÁÉÍÓÚÀÈËÏÖÜ][a-záéíóúàèëïöü]+(?:\s+\S+)+', k_schoon):
                namen.add(k_schoon)

    # === STUDENTNUMMERS: s/S-prefix of 7-cijferig getal ===
    nrs = re.findall(r'\b[sS]\d{6,8}\b|\b\d{7}\b', tekst)
    studentnrs.update(nrs)

    # === OPTIONELE FALLBACK: SpaCy NER op eerste 3000 tekens ===
    if SPACY_BESCHIKBAAR:
        doc = nlp(tekst[:3000])
        for ent in doc.ents:
            if ent.label_ == "PER" and len(ent.text.split()) >= 2:
                namen.add(ent.text)

    

    return namen, studentnrs


def anonimiseer_tekst_v2(tekst):
    """
    Anonimiseert tekst door namen en studentnummers te vervangen.
    Verwerkt ook namen die door PDF-extractie over twee regels zijn gebroken.

    Returns:
        dict met:
        - tekst: geanonimiseerde tekst
        - gevonden_namen: gedetecteerde namen
        - gevonden_studentnummers: gedetecteerde studentnummers
    """
    namen, studentnrs = extract_namen_structureel(tekst)

    log = {
        "gevonden_namen": sorted(list(namen)),
        "gevonden_studentnummers": sorted(list(studentnrs))
    }

    # Stap 1: studentnummers vervangen
    for nr in sorted(studentnrs, key=len, reverse=True):
        tekst = re.sub(re.escape(nr), '[STUDENTNR]', tekst)

    # Stap 2: namen vervangen, lang naar kort
    for naam in sorted(namen, key=len, reverse=True):
        # normale vervanging
        tekst = re.sub(r'\b' + re.escape(naam) + r'\b', '[NAAM]', tekst)
        
    # Stap 3: losse voornamen en achternamen vervangen
    stopwoorden = {'van', 'de', 'den', 'der', 'te', 'ten', 'Al', 'al', 'El', 'el'}
    naamdelen = set()
    for naam in namen:
        for deel in naam.split():
            if deel not in stopwoorden and len(deel) >= 3:
                naamdelen.add(deel)

    for deel in sorted(naamdelen, key=len, reverse=True):
        tekst = re.sub(r'\b' + re.escape(deel) + r'\b', '[NAAM]', tekst)
        # vervanging wanneer naam door PDF-extractie over twee regels is gebroken
        naam_gesplitst = naam.replace(' ', '\n')
        tekst = tekst.replace(naam_gesplitst, '[NAAM]')

    return {
        "tekst": tekst,
        "gevonden_namen": log["gevonden_namen"],
        "gevonden_studentnummers": log["gevonden_studentnummers"]
    }


# Testcel
test_tekst = "Jan de Vries (s1234567) heeft het Plan van Aanpak ingeleverd. Samen met Lisa Bakker."
test_resultaat = anonimiseer_tekst_v2(test_tekst)

print(f"Origineel:      {test_tekst}")
print(f"Geanonimiseerd: {test_resultaat['tekst']}")
print(f"Gevonden namen: {test_resultaat['gevonden_namen']}")
print(f"Gevonden nrs:   {test_resultaat['gevonden_studentnummers']}")

SpaCy geladen als aanvullende fallback.
Origineel:      Jan de Vries (s1234567) heeft het Plan van Aanpak ingeleverd. Samen met Lisa Bakker.
Geanonimiseerd: [NAAM] ([STUDENTNR]) heeft het Plan van Aanpak ingeleverd. Samen met Lisa Bakker.
Gevonden namen: ['Jan de Vries']
Gevonden nrs:   ['s1234567']


### 1.2 — Automatische anonimisering met structurele patroonherkenning

We detecteren namen en studentnummers via een combinatie van drie structurele regex-patronen en een optionele SpaCy NER-fallback.

**Waarom niet alleen SpaCy?**  
SpaCy's Nederlandse NER-model heeft zinscontext nodig. Tabelregels zoals `Artem Stasyuk - s1148519` of kommalijsten zoals `Auteur(s): Rami Al-Muhana, Sem Bersee` zijn geen zinnen, waardoor SpaCy ze niet betrouwbaar herkent. De structurele patronen zijn daarom de primaire detectiemethode.

**Drie patronen afgestemd op de documentformaten:**
- **Patroon 1 (B11-formaat):** `Voornaam Achternaam - s1234567`
- **Patroon 2 (C8-formaat):** `Voornaam Achternaam (1234567)` of `(s1234567)`
- **Patroon 3 (A2-formaat):** `Auteur(s): Naam1, Naam2, ...` — ook als de lijst over meerdere regels loopt

**Aanvullende fallbacks:**
- SpaCy NER wordt toegepast op de eerste 3000 tekens van de tekst wanneer het model beschikbaar is
- Een regex-patroon voor tweeledige hoofdletternamen vangt resterende namen op in lopende tekst, met een uitsluitlijst voor veelvoorkomende false positives

**Regelafbreking:** namen die door PDF-tabelextractie over twee regels zijn gebroken (bijv. `Bas\nKortenoever`) worden ook correct vervangen.

**Resultaat:** een functie `anonimiseer_tekst_v2(tekst)` die een dictionary teruggeeft met de geanonimiseerde tekst, de gevonden namen en de gevonden studentnummers.

### 1.3 — Alle documenten verwerken
We definiëren de zes PvA-documenten per groep, laden ze in, extraheren de tekst, en anonimiseren ze. Het resultaat is een dictionary met schone teksten per groep, klaar voor de LLM.

In [28]:
# Cel 1.3 — Alle PvA-documenten inladen, extraheren en anonimiseren

import os

# Documentenstructuur: per groep twee versies
documenten = {
    "B11": [
        "/Users/user/Desktop/POC/INDATAD/Groep B11/GroepB11_PvA_Tussentijds_Goed_2.pdf",
        "/Users/user/Desktop/POC/INDATAD/Groep B11/GroepB11_Plan_van_aanpak_1.pdf"
    ],
    "C8": [
        "/Users/user/Desktop/POC/INDATAD/Groep C8/GroepC8_PvA_Tussentijds_Voldoende_2.pdf",
        "/Users/user/Desktop/POC/INDATAD/Groep C8/GroepC8_Plan_van_Aanpak_1.pdf"
    ],
    "A2": [
        "/Users/user/Desktop/POC/INDATAD/Groep A2/GroepA2_PvA_Tussentijds_Onvoldoende_1.pdf",
        "/Users/user/Desktop/POC/INDATAD/Groep A2/GroepA2_Plan_van_aanpak_2.pdf"
    ]
}

# Verwerk alle documenten
verwerkte_documenten = {}
alle_afbeelding_logs = []

for groep, bestanden in documenten.items():
    verwerkte_documenten[groep] = []
    
    for pdf_pad in bestanden:
        bestandsnaam = os.path.basename(pdf_pad)
        print(f"Verwerken: {bestandsnaam}...")
        
        # Extra controle
        if not os.path.exists(pdf_pad):
            print(f"  FOUT: Bestand niet gevonden: {pdf_pad}")
            continue
        
        # Stap 1: tekst extraheren
        extractie = extract_pdf_tekst(pdf_pad)
        
        if not extractie["tekst"]:
            print(f"  WAARSCHUWING: Geen tekst gevonden in {bestandsnaam}")
            continue
        
        # Stap 2: afbeeldingen loggen
        if extractie["afbeelding_log"]:
            for log in extractie["afbeelding_log"]:
                print(f"  {log['melding']}")
                alle_afbeelding_logs.append({
                    "groep": groep,
                    "bestand": bestandsnaam,
                    **log
                })
        
        # Stap 3: anonimiseren
        anoniem = anonimiseer_tekst_v2(extractie["tekst"])
        
        if anoniem["gevonden_namen"]:
            print(f"  Geanonimiseerd: {len(anoniem['gevonden_namen'])} naam/namen verwijderd")
        if anoniem["gevonden_studentnummers"]:
            print(f"  Geanonimiseerd: {len(anoniem['gevonden_studentnummers'])} studentnummer(s) verwijderd")
        
        # Opslaan
        verwerkte_documenten[groep].append({
            "bestandsnaam": bestandsnaam,
            "origineel_paginas": extractie["metadata"]["aantal_paginas"],
            "geanonimiseerde_tekst": anoniem["tekst"],
            "anonimisatie_log": {
                "namen": anoniem["gevonden_namen"],
                "studentnummers": anoniem["gevonden_studentnummers"]
            },
            "afbeelding_log": extractie["afbeelding_log"]
        })
        
        print(f"  Klaar ({extractie['metadata']['aantal_paginas']} pagina's)")
    print()

# Samenvatting
print("=" * 50)
print("SAMENVATTING")
print("=" * 50)

for groep, docs in verwerkte_documenten.items():
    print(f"\nGroep {groep}: {len(docs)} document(en) verwerkt")
    for doc in docs:
        print(f"  - {doc['bestandsnaam']} ({doc['origineel_paginas']} pagina's)")

if alle_afbeelding_logs:
    print(f"\nAfbeeldingen gedetecteerd op {len(alle_afbeelding_logs)} pagina('s). Deze zijn niet verwerkt.")

Verwerken: GroepB11_PvA_Tussentijds_Goed_2.pdf...
  Geanonimiseerd: 5 naam/namen verwijderd
  Geanonimiseerd: 5 studentnummer(s) verwijderd
  Klaar (13 pagina's)
Verwerken: GroepB11_Plan_van_aanpak_1.pdf...
  Pagina 9 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 10 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 16 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 17 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 19 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 23 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 25 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 39 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 40 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de LLM.
  Pagina 42 bevat 1 afbeelding(en) die niet verwerkt kunnen worden door de

### 1.4 — Geanonimiseerde teksten en logs opslaan

We slaan alle geanonimiseerde teksten op als JSON, zodat we in latere secties direct kunnen laden zonder de PDF-pipeline opnieuw te draaien. Het afbeeldingslog wordt apart opgeslagen.

In [24]:
# Cel 1.4 — Geanonimiseerde teksten en logs opslaan

# Maak een output map aan voor geanonimiseerde data
GEANON_DIR = os.path.join(OUTPUT_DIR, "geanonimiseerd")
os.makedirs(GEANON_DIR, exist_ok=True)

# === 1: Sla alles op in één JSON bestand ===
opslag_pad = os.path.join(GEANON_DIR, "verwerkte_documenten.json")
with open(opslag_pad, "w", encoding="utf-8") as f:
    json.dump(verwerkte_documenten, f, ensure_ascii=False, indent=2)
print(f"Alle geanonimiseerde documenten opgeslagen: {opslag_pad}")

# === 2: Sla ook per document een los tekstbestand op (voor inzicht) ===
for groep, docs in verwerkte_documenten.items():
    for doc in docs:
        bestandsnaam = doc["bestandsnaam"].replace(".pdf", "")
        txt_pad = os.path.join(GEANON_DIR, f"{groep}_{bestandsnaam}.txt")
        with open(txt_pad, "w", encoding="utf-8") as f:
            f.write(doc["geanonimiseerde_tekst"])
        print(f"  Opgeslagen: {txt_pad}")

# === 3: Afbeelding-log opslaan ===
afbeelding_log_pad = os.path.join(LOG_DIR, "afbeelding_log.json")
with open(afbeelding_log_pad, "w", encoding="utf-8") as f:
    json.dump(alle_afbeelding_logs, f, ensure_ascii=False, indent=2)

print(f"\nAfbeelding-log opgeslagen: {afbeelding_log_pad}")
print(f"Totaal gemiste afbeeldingen: {sum(log['aantal_afbeeldingen'] for log in alle_afbeelding_logs)}")
print(f"\nJe kunt nu in latere secties laden vanuit: {opslag_pad}")

Alle geanonimiseerde documenten opgeslagen: output/geanonimiseerd/verwerkte_documenten.json
  Opgeslagen: output/geanonimiseerd/B11_GroepB11_PvA_Tussentijds_Goed_2.txt
  Opgeslagen: output/geanonimiseerd/B11_GroepB11_Plan_van_aanpak_1.txt
  Opgeslagen: output/geanonimiseerd/C8_GroepC8_PvA_Tussentijds_Voldoende_2.txt
  Opgeslagen: output/geanonimiseerd/C8_GroepC8_Plan_van_Aanpak_1.txt
  Opgeslagen: output/geanonimiseerd/A2_GroepA2_PvA_Tussentijds_Onvoldoende_1.txt
  Opgeslagen: output/geanonimiseerd/A2_GroepA2_Plan_van_aanpak_2.txt

Afbeelding-log opgeslagen: logs/afbeelding_log.json
Totaal gemiste afbeeldingen: 23

Je kunt nu in latere secties laden vanuit: output/geanonimiseerd/verwerkte_documenten.json


### 1.5 — Verificatie: lek-check op anonimisering
Controleert automatisch of er nog bekende namen of naamdelen voorkomen in de geanonimiseerde teksten. Toont groen (✅) of rood (⚠️) per document.

In [30]:
# Cel 1.5 — Verificatie: lek-check op anonimisering

def verificatie_anonimisering(verwerkte_documenten):
    """
    Controleert per document of er nog namen of naamdelen
    voorkomen in de geanonimiseerde tekst.
    """
    stopwoorden = {'van', 'de', 'den', 'der', 'te', 'ten', 'al', 'el'}
    totaal_docs = 0
    totaal_schoon = 0

    for groep, docs in verwerkte_documenten.items():
        for doc in docs:
            totaal_docs += 1
            geanon = doc["geanonimiseerde_tekst"]
            namen = doc["anonimisatie_log"]["namen"]
            lekken = []

            for naam in namen:
                # Check volledige naam
                if naam in geanon:
                    lekken.append(f"volledige naam '{naam}'")

                # Check losse naamdelen
                for deel in naam.split():
                    if deel.lower() in stopwoorden or len(deel) < 3:
                        continue
                    hits = len(re.findall(r'\b' + re.escape(deel) + r'\b', geanon))
                    if hits > 0:
                        lekken.append(f"'{deel}' ({hits}x)")

            if lekken:
                print(f"⚠️  Groep {groep} | {doc['bestandsnaam']}")
                for lek in lekken:
                    print(f"      → {lek}")
            else:
                print(f"✅ Groep {groep} | {doc['bestandsnaam']}")
                totaal_schoon += 1

    print(f"\n{'='*50}")
    print(f"RESULTAAT: {totaal_schoon}/{totaal_docs} documenten volledig geanonimiseerd")
    if totaal_schoon == totaal_docs:
        print("Alle documenten zijn schoon. Klaar voor de LLM.")
    else:
        print("Pas cel 1.2 aan en draai cel 1.3 + 1.5 opnieuw.")

verificatie_anonimisering(verwerkte_documenten)

✅ Groep B11 | GroepB11_PvA_Tussentijds_Goed_2.pdf
✅ Groep B11 | GroepB11_Plan_van_aanpak_1.pdf
✅ Groep C8 | GroepC8_PvA_Tussentijds_Voldoende_2.pdf
✅ Groep C8 | GroepC8_Plan_van_Aanpak_1.pdf
✅ Groep A2 | GroepA2_PvA_Tussentijds_Onvoldoende_1.pdf
✅ Groep A2 | GroepA2_Plan_van_aanpak_2.pdf

RESULTAAT: 6/6 documenten volledig geanonimiseerd
Alle documenten zijn schoon. Klaar voor de LLM.


---
## Sectie 2 — DV1: Evaluatiekader voor Effectieve Formatieve Feedback

**Deelvraag 1:** Wat zijn de kenmerken van effectieve en consistente formatieve feedback binnen het hoger onderwijs?

In deze sectie wordt geen uitgebreide literatuurtekst geschreven. In plaats daarvan worden de kenmerken van effectieve formatieve feedback — gebaseerd op het onderzoeksvoorstel en literatuur over feedup/feedback/feedforward (Hattie & Timperley, 2007) — omgezet naar een operationeel evaluatiekader in Python dictionaries.

Dit kader definieert zeven meetbare dimensies. Elke dimensie heeft een beschrijving, concrete indicatoren en een scoreschaal van 1 tot 3. Het kader functioneert als benchmarkinstrument in DV4, waar de LLM-output wordt vergeleken met de docentfeedback als gouden standaard.

### 2.1 — Definitie van de zeven feedbackdimensies

De zeven dimensies zijn gebaseerd op het drie-niveaumodel van Hattie & Timperley (feedup, feedback, feedforward) aangevuld met kwaliteitscriteria die relevant zijn voor rubric-gebaseerde feedback in het hoger onderwijs.

In [13]:
# Cel 2.1 — Evaluatiekader: zeven dimensies van effectieve formatieve feedback

EVALUATIEKADER = {

    "feedup": {
        "beschrijving": (
            "In welke mate maakt de feedback duidelijk wat het doel of "
            "de verwachting was (waar moest het naartoe)?"
        ),
        "indicatoren": [
            "De feedback verwijst expliciet naar het leerdoel of de opdrachtomschrijving",
            "De feedback benoemt wat van studenten verwacht werd",
            "Er is een duidelijke koppeling met de rubric of beoordelingscriterium"
        ],
        "scoreschaal": {
            1: "Doel of verwachting wordt niet benoemd",
            2: "Doel of verwachting wordt gedeeltelijk benoemd",
            3: "Doel of verwachting wordt expliciet en concreet benoemd"
        }
    },

    "feedback": {
        "beschrijving": (
            "In welke mate beschrijft de feedback de huidige prestatie "
            "ten opzichte van het doel (hoe staat het ervoor)?"
        ),
        "indicatoren": [
            "De feedback beschrijft wat de student heeft gedaan",
            "Er is een oordeel of beoordeling van de huidige kwaliteit",
            "De feedback is gebaseerd op aantoonbare elementen uit het document"
        ],
        "scoreschaal": {
            1: "Geen beschrijving van de huidige prestatie",
            2: "Globale beschrijving zonder onderbouwing",
            3: "Concrete beschrijving gebaseerd op het werk van de student"
        }
    },

    "feedforward": {
        "beschrijving": (
            "In welke mate geeft de feedback richting voor verbetering "
            "(wat is de volgende stap)?"
        ),
        "indicatoren": [
            "De feedback bevat concrete verbeterpunten of aanbevelingen",
            "De suggesties zijn uitvoerbaar voor de student",
            "Er wordt aangegeven hoe de student zich kan verbeteren"
        ],
        "scoreschaal": {
            1: "Geen verbeterpunten of vervolgstappen",
            2: "Algemene suggesties zonder concreetheid",
            3: "Concrete, uitvoerbare verbeterpunten gericht op de volgende stap"
        }
    },

    "specificiteit": {
        "beschrijving": (
            "In welke mate is de feedback specifiek en herleidbaar "
            "tot het ingediende werk?"
        ),
        "indicatoren": [
            "De feedback verwijst naar specifieke onderdelen of passages",
            "Er worden concrete voorbeelden gegeven",
            "De feedback is niet generiek of van toepassing op elk document"
        ],
        "scoreschaal": {
            1: "Volledig generiek, niet herleidbaar tot het werk",
            2: "Gedeeltelijk specifiek, soms onderbouwd",
            3: "Volledig specifiek en direct herleidbaar tot het ingediende werk"
        }
    },

    "rubric_aansluiting": {
        "beschrijving": (
            "In welke mate sluit de feedback aan op de criteria "
            "uit de rubric?"
        ),
        "indicatoren": [
            "De feedback dekt de rubriccriteria die van toepassing zijn",
            "De terminologie uit de rubric wordt gebruikt of herkend",
            "Er worden geen criteria beoordeeld die buiten de rubric vallen"
        ],
        "scoreschaal": {
            1: "Geen aansluiting met de rubric",
            2: "Gedeeltelijke aansluiting, sommige criteria worden gemist",
            3: "Volledige aansluiting met de relevante rubriccriteria"
        }
    },

    "constructieve_toon": {
        "beschrijving": (
            "In welke mate is de toon van de feedback constructief, "
            "respectvol en motiverend?"
        ),
        "indicatoren": [
            "De feedback is niet afbrekend of negatief geformuleerd",
            "Positieve elementen worden erkend naast verbeterpunten",
            "De toon nodigt uit tot verbetering in plaats van te ontmoedigen"
        ],
        "scoreschaal": {
            1: "Toon is afbrekend, negatief of demotiverend",
            2: "Neutrale toon, maar weinig erkenning of motivatie",
            3: "Constructieve en respectvolle toon die uitnodigt tot verbetering"
        }
    },

    "consistentie": {
        "beschrijving": (
            "In welke mate is de feedback consistent over vergelijkbare "
            "documenten of criteria (bij meerdere beoordelingen)?"
        ),
        "indicatoren": [
            "Vergelijkbare documenten krijgen vergelijkbare feedback",
            "De diepgang en lengte van feedback is gelijkmatig verdeeld",
            "Dezelfde rubriccriteria worden op gelijke wijze toegepast"
        ],
        "scoreschaal": {
            1: "Grote inconsistenties tussen vergelijkbare beoordelingen",
            2: "Gedeeltelijk consistent, merkbare verschillen",
            3: "Consistent toegepast over vergelijkbare documenten en criteria"
        }
    }
}

print(f"Evaluatiekader geladen met {len(EVALUATIEKADER)} dimensies:")
for dimensie in EVALUATIEKADER:
    print(f"  - {dimensie}")

Evaluatiekader geladen met 7 dimensies:
  - feedup
  - feedback
  - feedforward
  - specificiteit
  - rubric_aansluiting
  - constructieve_toon
  - consistentie


### 2.2 — Scorefunctie en overzicht van het kader

De scorefunctie maakt het kader bruikbaar als meetinstrument. In DV4 worden scores per dimensie toegekend aan zowel de LLM-output als de docentfeedback, waarna de resultaten worden vergeleken.

In [14]:
# Cel 2.2 — Scorefunctie en weergave van het evaluatiekader

def score_feedback(dimensie, score, toelichting=""):
    """
    Legt een score vast voor één feedbackdimensie.

    Parameters:
        dimensie    : naam van de dimensie (sleutel uit EVALUATIEKADER)
        score       : integer 1, 2 of 3
        toelichting : optionele motivatie bij de score

    Returns:
        dict met dimensie, score, label en toelichting
    """
    if dimensie not in EVALUATIEKADER:
        raise ValueError(f"Onbekende dimensie: '{dimensie}'. "
                         f"Kies uit: {list(EVALUATIEKADER.keys())}")
    if score not in (1, 2, 3):
        raise ValueError("Score moet 1, 2 of 3 zijn.")

    label = EVALUATIEKADER[dimensie]["scoreschaal"][score]

    return {
        "dimensie": dimensie,
        "score": score,
        "label": label,
        "toelichting": toelichting
    }


def toon_kader_overzicht():
    """Toont een overzicht van alle dimensies met beschrijving en scoreschaal."""
    print("=" * 60)
    print("EVALUATIEKADER — Effectieve Formatieve Feedback")
    print("=" * 60)
    for naam, inhoud in EVALUATIEKADER.items():
        print(f"\n{naam.upper().replace('_', ' ')}")
        print(f"  {inhoud['beschrijving']}")
        print(f"  Scoreschaal:")
        for punt, label in inhoud["scoreschaal"].items():
            print(f"    {punt} — {label}")

toon_kader_overzicht()

EVALUATIEKADER — Effectieve Formatieve Feedback

FEEDUP
  In welke mate maakt de feedback duidelijk wat het doel of de verwachting was (waar moest het naartoe)?
  Scoreschaal:
    1 — Doel of verwachting wordt niet benoemd
    2 — Doel of verwachting wordt gedeeltelijk benoemd
    3 — Doel of verwachting wordt expliciet en concreet benoemd

FEEDBACK
  In welke mate beschrijft de feedback de huidige prestatie ten opzichte van het doel (hoe staat het ervoor)?
  Scoreschaal:
    1 — Geen beschrijving van de huidige prestatie
    2 — Globale beschrijving zonder onderbouwing
    3 — Concrete beschrijving gebaseerd op het werk van de student

FEEDFORWARD
  In welke mate geeft de feedback richting voor verbetering (wat is de volgende stap)?
  Scoreschaal:
    1 — Geen verbeterpunten of vervolgstappen
    2 — Algemene suggesties zonder concreetheid
    3 — Concrete, uitvoerbare verbeterpunten gericht op de volgende stap

SPECIFICITEIT
  In welke mate is de feedback specifiek en herleidbaar tot

### 2.3 — Validatie van het kader met een voorbeeldscore

Als sanity check kennen we handmatig een voorbeeldscore toe aan één dimensie. Dit bevestigt dat de scorefunctie correct werkt en klaar is voor gebruik in DV4.

In [15]:
# Cel 2.3 — Validatie: handmatige voorbeeldscore

voorbeeld = score_feedback(
    dimensie="feedforward",
    score=2,
    toelichting=(
        "De feedback geeft globaal aan dat de probleemstelling "
        "verduidelijkt moet worden, maar noemt geen concrete aanpak."
    )
)

print("Voorbeeldscore:")
for k, v in voorbeeld.items():
    print(f"  {k}: {v}")

print("\nEvaluatiekader is gevalideerd en klaar voor gebruik in DV4.")

Voorbeeldscore:
  dimensie: feedforward
  score: 2
  label: Algemene suggesties zonder concreetheid
  toelichting: De feedback geeft globaal aan dat de probleemstelling verduidelijkt moet worden, maar noemt geen concrete aanpak.

Evaluatiekader is gevalideerd en klaar voor gebruik in DV4.


---
## Sectie 3 — DV2: Rubric naar Prompttemplates

**Deelvraag 2:** Hoe kan een rubric worden omgezet naar een effectieve prompt zodat de LLM bruikbare formatieve feedback genereert?

In deze sectie vertalen we de rubric (Toets 2, I-An 2.1 — Infrastructuur Analyseren) naar prompttemplates. We ontwerpen twee promptvarianten:

- **Variant A (zero-shot):** een gestructureerde prompt met de rubriccriteria, zonder voorbeeldfeedback
- **Variant B (few-shot):** dezelfde prompt aangevuld met een voorbeeld van goede formatieve feedback

Per rubriccriterium wordt een aparte prompt gebouwd, plus één overkoepelende prompt voor de hele rubric. We testen beide varianten op dezelfde geanonimiseerde oplevering en vergelijken de output op rubric-aansluiting, specificiteit en bruikbaarheid.

### 3.0 — Geanonimiseerde documenten laden

In plaats van de PDF-pipeline opnieuw te draaien, laden we de eerder opgeslagen geanonimiseerde teksten. Dit maakt het mogelijk om Sectie 3 (en later 4, 5, 6) onafhankelijk van Sectie 1 te draaien.

In [25]:
# Cel 3.0 — Geanonimiseerde documenten laden vanuit JSON

GEANON_PAD = os.path.join(OUTPUT_DIR, "geanonimiseerd", "verwerkte_documenten.json")

if os.path.exists(GEANON_PAD):
    with open(GEANON_PAD, "r", encoding="utf-8") as f:
        verwerkte_documenten = json.load(f)

    print(f"Geanonimiseerde documenten geladen vanuit: {GEANON_PAD}")
    for groep, docs in verwerkte_documenten.items():
        print(f"  Groep {groep}: {len(docs)} document(en)")
        for doc in docs:
            print(f"    - {doc['bestandsnaam']} ({len(doc['geanonimiseerde_tekst'])} tekens)")
else:
    print("FOUT: Geen opgeslagen data gevonden.")
    print("Draai eerst Sectie 1 (cel 1.1 t/m 1.4) om de documenten te verwerken.")


Geanonimiseerde documenten geladen vanuit: output/geanonimiseerd/verwerkte_documenten.json
  Groep B11: 2 document(en)
    - GroepB11_PvA_Tussentijds_Goed_2.pdf (18118 tekens)
    - GroepB11_Plan_van_aanpak_1.pdf (86446 tekens)
  Groep C8: 2 document(en)
    - GroepC8_PvA_Tussentijds_Voldoende_2.pdf (10459 tekens)
    - GroepC8_Plan_van_Aanpak_1.pdf (90713 tekens)
  Groep A2: 2 document(en)
    - GroepA2_PvA_Tussentijds_Onvoldoende_1.pdf (4079 tekens)
    - GroepA2_Plan_van_aanpak_2.pdf (61511 tekens)


### 3.1 — Rubric-criteria als Python dictionary

De beoordelingscriteria voor Toets 2 (I-An 2.1 en I-An 2.2) zijn handmatig gestructureerd op basis van de toetsmatrijs PDF. De criteria worden als dictionary opgeslagen zodat ze direct in de prompttemplates kunnen worden ingevuld. Er wordt hier geen PDF gelezen — de rubric is eenmalig vertaald naar code.

In [16]:
# Cel 3.1 — Rubric-criteria voor Toets 2 (I-An 2.1): Plan van Aanpak

# Handmatig gestructureerd op basis van de toetsmatrijs PDF
# Bron: i_INDATAD_tm_toets1&toets2_1.0 (3).pdf — pagina 4-5

RUBRIC_CRITERIA = {
    "I-An 2.1": {
        "naam": "Methodisch onderzoek en analyse",
        "weging": "35%",
        "beschrijving": (
            "De student voert op methodische wijze een relevant onderzoek op het gebied "
            "van infrastructuur voor een datavraagstuk uit. Deze analyse gebeurt op basis "
            "van een geformuleerde relevante en afgebakende probleemstelling die aansluit "
            "op de opdracht."
        ),
        "vereiste_onderdelen": [
            "Context",
            "Beschrijving van de organisatie",
            "Huidige situatie / bedrijfsproces",
            "Gewenste situatie",
            "Probleemstelling",
            "Scope beschrijving",
            "Requirementsanalyse",
            "Risicolijst",
            "Oplossingsrichtingen",
            "Planning",
        ],
        "beoordelingscriteria": {
            "voldoende": (
                "De student analyseert de technische vraagstukken die aansluiten op de "
                "probleemstelling behorende bij de opdracht. De student analyseert "
                "relevante betrokken organisatie(s) en omgevingsfactoren en licht toe "
                "waarom deze relevant zijn in de context van de opdracht. De student "
                "verwerkt deze betrokkenen en omgevingsfactoren in de requirementsanalyse. "
                "De student bewijst in een plan van aanpak zijn/haar bekwaamheid om een "
                "data-gerelateerde opdracht op methodische wijze aan te pakken."
            )
        }
    },
    "I-An 2.2": {
        "naam": "Onderbouwing methoden en technieken",
        "weging": "25%",
        "beschrijving": (
            "De student onderbouwt de geschiktheid van de gebruikte methoden en "
            "technieken op basis van de probleemstelling, waarbij de tekortkomingen "
            "van de gekozen methoden en technieken ook worden benoemd."
        ),
        "beoordelingscriteria": {
            "voldoende": (
                "De student onderbouwt de geschiktheid en beperkingen van gekozen methoden "
                "en technieken en stuurt de dataverzameling bij wanneer nodig. De student "
                "houdt op gestructureerde wijze product- en projectrisico's bij en anticipeert "
                "hierop. De student komt op een methodische en inzichtelijke manier tot een "
                "onderbouwd TOGAF model passend bij de projectopdracht."
            )
        }
    }
}

print(f"Rubric geladen met {len(RUBRIC_CRITERIA)} criteria:")
for code, inhoud in RUBRIC_CRITERIA.items():
    print(f"  {code}: {inhoud['naam']} (weging: {inhoud['weging']})")
    if "vereiste_onderdelen" in inhoud:
        print(f"         Vereiste onderdelen: {len(inhoud['vereiste_onderdelen'])}")

Rubric geladen met 2 criteria:
  I-An 2.1: Methodisch onderzoek en analyse (weging: 35%)
         Vereiste onderdelen: 10
  I-An 2.2: Onderbouwing methoden en technieken (weging: 25%)


### 3.2 — Helperfunctie: Ollama aanroepen via Python requests

Een herbruikbare functie die een prompt naar Ollama stuurt via `localhost:11434`. Ondersteunt twee modi: streaming (output token voor token zichtbaar) en niet-streaming. Elke aanroep wordt gelogd. Streaming is standaard aan voor grote prompts zodat je direct ziet dat het model reageert.

In [22]:
# Cel 3.2 — Helperfunctie: Ollama aanroepen met streaming

def ollama_aanroep(prompt, model=MODEL_NAME, temperatuur=0.3,
                   max_tokens=512, stream=True):
    """
    Stuurt een prompt naar Ollama en logt het resultaat.

    Parameters:
        prompt      : volledige prompt als string
        model       : modelnaam
        temperatuur : 0.0 = deterministisch, 1.0 = creatief
        max_tokens  : max output tokens (houd laag voor snelheid)
        stream      : True = output token voor token zichtbaar

    Returns:
        dict met response, model, temperatuur, duur_seconden, timestamp
    """
    timestamp = datetime.now().isoformat()

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": {
            "temperature": temperatuur,
            "num_predict": max_tokens
        }
    }

    try:
        start = datetime.now()

        if stream:
            # Streaming: print tokens direct terwijl ze binnenkomen
            response = requests.post(
                f"{OLLAMA_BASE_URL}/api/generate",
                json=payload,
                stream=True,
                timeout=1800  # 30 minuten max
            )
            volledige_response = ""
            print("Model reageert (streaming):\n" + "-"*40)

            for regel in response.iter_lines():
                if regel:
                    data = json.loads(regel.decode("utf-8"))
                    token = data.get("response", "")
                    print(token, end="", flush=True)
                    volledige_response += token
                    if data.get("done", False):
                        break

            print("\n" + "-"*40)
            duur = (datetime.now() - start).total_seconds()

        else:
            # Niet-streaming: wacht op volledige response
            response = requests.post(
                f"{OLLAMA_BASE_URL}/api/generate",
                json=payload,
                timeout=1800
            )
            data = response.json()
            volledige_response = data.get("response", "")
            duur = (datetime.now() - start).total_seconds()

        resultaat = {
            "response": volledige_response,
            "model": model,
            "temperatuur": temperatuur,
            "duur_seconden": round(duur, 2),
            "timestamp": timestamp,
            "prompt_lengte": len(prompt),
            "response_lengte": len(volledige_response),
        }

        # Log naar bestand
        log_entry = {
            "prompt": prompt[:300] + "..." if len(prompt) > 300 else prompt,
            **resultaat
        }
        log_pad = os.path.join(LOG_DIR, "ollama_aanroepen.jsonl")
        with open(log_pad, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

        print(f"\nDuur: {resultaat['duur_seconden']}s | "
              f"Response: {resultaat['response_lengte']} tekens")
        return resultaat

    except requests.Timeout:
        print("FOUT: Timeout — model reageert niet binnen 30 minuten.")
        return None
    except requests.ConnectionError:
        print("FOUT: Kan geen verbinding maken met Ollama op localhost:11434")
        return None


# Test
test = ollama_aanroep(
    "Geef in maximaal drie zinnen uitleg over wat een Plan van Aanpak is.",
    max_tokens=100,
    stream=True
)

Model reageert (streaming):
----------------------------------------
Een Plan van Aanpak is een strategische documentatie die de stappen en acties beschrijft om een bepaald doel of probleem te realiseren. Het bevat de volgorde, verantwoordelijkheden en middelen voor elke stap in het proces. Dit plan helpt bij het organiseren en structureren van taken en resourcenten voor effectieve uitvoering.
----------------------------------------

Duur: 121.06s | Response: 327 tekens


### 3.3 — Promptvariant A: Zero-shot per rubriccriterium

De zero-shot prompt bevat de rubricbeschrijving, de vereiste onderdelen en een gestructureerd outputformat. De LLM krijgt geen voorbeeldfeedback. Per criterium wordt een aparte prompt gebouwd.

In [18]:
# Cel 3.3 — Promptvariant A: zero-shot

def bouw_prompt_zero_shot(criterium_code, student_tekst):
    """
    Bouwt een zero-shot prompt voor één rubriccriterium.

    Parameters:
        criterium_code : sleutel uit RUBRIC_CRITERIA (bijv. "I-An 2.1")
        student_tekst  : geanonimiseerde tekst van het studentdocument

    Returns:
        prompt als string
    """
    criterium = RUBRIC_CRITERIA[criterium_code]

    vereisten = ""
    if "vereiste_onderdelen" in criterium:
        vereisten = "\n".join(
            f"  - {onderdeel}" for onderdeel in criterium["vereiste_onderdelen"]
        )
        vereisten = f"\nVereiste onderdelen in het document:\n{vereisten}\n"

    prompt = f"""Je bent een ervaren HBO-docent Informatica aan Hogeschool Leiden. Je geeft formatieve feedback op een Plan van Aanpak van studenten.

RUBRICCRITERIUM: {criterium_code} — {criterium['naam']}
Weging: {criterium['weging']}

Beschrijving van het criterium:
{criterium['beschrijving']}

Beoordelingscriteria (niveau voldoende):
{criterium['beoordelingscriteria']['voldoende']}
{vereisten}
INSTRUCTIES:
- Geef formatieve feedback, GEEN cijfer of beoordeling.
- Gebruik de volgende structuur voor je feedback:

  FEEDUP (wat was het doel):
  [Beschrijf kort welk doel of welke verwachting bij dit criterium hoort]

  FEEDBACK (hoe staat het ervoor):
  [Beschrijf concreet wat de student goed heeft gedaan en waar tekortkomingen zitten, met verwijzingen naar specifieke onderdelen uit het document]

  FEEDFORWARD (wat is de volgende stap):
  [Geef concrete, uitvoerbare verbeterpunten die de student kan oppakken]

- Schrijf in het Nederlands.
- Wees specifiek: verwijs naar concrete onderdelen uit het document.
- Wees constructief: benoem ook wat goed gaat.

STUDENTDOCUMENT:
{student_tekst}

Geef nu je formatieve feedback op basis van criterium {criterium_code}:"""

    return prompt


# Toon een voorbeeld van de prompt (zonder studenttekst) om de structuur te checken
voorbeeld_prompt = bouw_prompt_zero_shot("I-An 2.1", "[hier komt de studenttekst]")
print(f"Promptlengte: {len(voorbeeld_prompt)} tekens")
print(f"\nEerste 800 tekens van de prompt:\n{'='*50}")
print(voorbeeld_prompt[:800])

Promptlengte: 2055 tekens

Eerste 800 tekens van de prompt:
Je bent een ervaren HBO-docent Informatica aan Hogeschool Leiden. Je geeft formatieve feedback op een Plan van Aanpak van studenten.

RUBRICCRITERIUM: I-An 2.1 — Methodisch onderzoek en analyse
Weging: 35%

Beschrijving van het criterium:
De student voert op methodische wijze een relevant onderzoek op het gebied van infrastructuur voor een datavraagstuk uit. Deze analyse gebeurt op basis van een geformuleerde relevante en afgebakende probleemstelling die aansluit op de opdracht.

Beoordelingscriteria (niveau voldoende):
De student analyseert de technische vraagstukken die aansluiten op de probleemstelling behorende bij de opdracht. De student analyseert relevante betrokken organisatie(s) en omgevingsfactoren en licht toe waarom deze relevant zijn in de context van de opdracht. De student v


### 3.4 — Promptvariant B: Few-shot met voorbeeldfeedback

De few-shot prompt is identiek aan variant A, maar bevat een concreet voorbeeld van goede formatieve feedback. Dit geeft de LLM een referentiekader voor toon, diepgang en structuur.

In [19]:
# Cel 3.4 — Promptvariant B: few-shot

VOORBEELD_FEEDBACK = {
    "I-An 2.1": """FEEDUP (wat was het doel):
Bij criterium I-An 2.1 wordt verwacht dat het Plan van Aanpak een methodische analyse bevat van het datavraagstuk. Het document moet een relevante en afgebakende probleemstelling bevatten, plus een analyse van de organisatie, betrokkenen en omgevingsfactoren, verwerkt in een requirementsanalyse.

FEEDBACK (hoe staat het ervoor):
De context is helder beschreven en geeft goed weer in welke setting het project plaatsvindt. De probleemstelling is aanwezig maar nog vrij breed geformuleerd — het is niet duidelijk welk specifiek data-infrastructuurprobleem wordt onderzocht. De requirementsanalyse bevat functionele en niet-functionele requirements, maar de koppeling met de probleemstelling ontbreekt. De risicolijst is aanwezig en bevat relevante risico's met mitigaties.

FEEDFORWARD (wat is de volgende stap):
1. Verscherp de probleemstelling: formuleer één concrete hoofdvraag die direct aansluit op het data-infrastructuurvraagstuk.
2. Koppel de requirements expliciet aan de probleemstelling — licht per requirement toe waarom deze relevant is voor het oplossen van het probleem.
3. Voeg een stakeholderanalyse toe waarin je toelicht welke betrokkenen welke belangen hebben bij de oplossing."""
}


def bouw_prompt_few_shot(criterium_code, student_tekst):
    """
    Bouwt een few-shot prompt: identiek aan zero-shot maar met een voorbeeld
    van goede formatieve feedback als referentie.
    """
    criterium = RUBRIC_CRITERIA[criterium_code]

    vereisten = ""
    if "vereiste_onderdelen" in criterium:
        vereisten = "\n".join(
            f"  - {onderdeel}" for onderdeel in criterium["vereiste_onderdelen"]
        )
        vereisten = f"\nVereiste onderdelen in het document:\n{vereisten}\n"

    voorbeeld = VOORBEELD_FEEDBACK.get(criterium_code, "")
    voorbeeld_sectie = ""
    if voorbeeld:
        voorbeeld_sectie = f"""
VOORBEELD van goede formatieve feedback op dit criterium (voor een ander document):
---
{voorbeeld}
---
Gebruik dit voorbeeld als referentie voor toon, diepgang en structuur. Pas de inhoud aan op het studentdocument hieronder.
"""

    prompt = f"""Je bent een ervaren HBO-docent Informatica aan Hogeschool Leiden. Je geeft formatieve feedback op een Plan van Aanpak van studenten.

RUBRICCRITERIUM: {criterium_code} — {criterium['naam']}
Weging: {criterium['weging']}

Beschrijving van het criterium:
{criterium['beschrijving']}

Beoordelingscriteria (niveau voldoende):
{criterium['beoordelingscriteria']['voldoende']}
{vereisten}
{voorbeeld_sectie}
INSTRUCTIES:
- Geef formatieve feedback, GEEN cijfer of beoordeling.
- Gebruik de volgende structuur voor je feedback:

  FEEDUP (wat was het doel):
  [Beschrijf kort welk doel of welke verwachting bij dit criterium hoort]

  FEEDBACK (hoe staat het ervoor):
  [Beschrijf concreet wat de student goed heeft gedaan en waar tekortkomingen zitten, met verwijzingen naar specifieke onderdelen uit het document]

  FEEDFORWARD (wat is de volgende stap):
  [Geef concrete, uitvoerbare verbeterpunten die de student kan oppakken]

- Schrijf in het Nederlands.
- Wees specifiek: verwijs naar concrete onderdelen uit het document.
- Wees constructief: benoem ook wat goed gaat.

STUDENTDOCUMENT:
{student_tekst}

Geef nu je formatieve feedback op basis van criterium {criterium_code}:"""

    return prompt


# Vergelijk promptlengtes
zs = bouw_prompt_zero_shot("I-An 2.1", "[studenttekst]")
fs = bouw_prompt_few_shot("I-An 2.1", "[studenttekst]")
print(f"Zero-shot promptlengte: {len(zs)} tekens")
print(f"Few-shot promptlengte:  {len(fs)} tekens")
print(f"Verschil: +{len(fs) - len(zs)} tekens door voorbeeldfeedback")

Zero-shot promptlengte: 2042 tekens
Few-shot promptlengte:  3484 tekens
Verschil: +1442 tekens door voorbeeldfeedback


### 3.5 — Test: beide promptvarianten op één document

We testen beide varianten op Groep B11, versie 1, criterium I-An 2.1. Streaming is aan zodat je direct output ziet. De max output is beperkt tot 300 tokens voor de test — genoeg om de structuur en kwaliteit te beoordelen.

In [23]:
# Cel 3.5 — Test: beide varianten op Groep B11 versie 1

def extraheer_relevante_secties(tekst, max_tekens=2500):
    """
    Slaat omslag, versiebeheer en inhoudsopgave over.
    Stuurt alleen inhoudelijke secties naar het model.
    """
    inhoud_starters = [
        "context", "inleiding", "organisatie", "probleemstelling",
        "huidige situatie", "gewenste situatie", "scope", "requirements",
        "risico", "oplossing", "planning", "stakeholder"
    ]

    regels = tekst.split('\n')
    relevante_regels = []
    inhoudsopgave_voorbij = False

    for regel in regels:
        regel_lower = regel.lower().strip()

        if any(x in regel_lower for x in ['versiebeheer', 'inhoudsopgave', 'versie datum']):
            inhoudsopgave_voorbij = False
            continue

        if any(regel_lower.startswith(x) for x in inhoud_starters):
            inhoudsopgave_voorbij = True

        if inhoudsopgave_voorbij:
            relevante_regels.append(regel)

    relevante_tekst = '\n'.join(relevante_regels)

    if len(relevante_tekst) < 500:
        kwart = len(tekst) // 4
        relevante_tekst = tekst[kwart:]

    if len(relevante_tekst) > max_tekens:
        relevante_tekst = (relevante_tekst[:max_tekens] +
                           "\n\n[... ingekort ...]")

    return relevante_tekst


# Testdocument
test_doc = verwerkte_documenten["B11"][0]
volledige_tekst = test_doc["geanonimiseerde_tekst"]
test_tekst = extraheer_relevante_secties(volledige_tekst, max_tekens=2500)

print(f"Document  : {test_doc['bestandsnaam']}")
print(f"Origineel : {len(volledige_tekst)} tekens")
print(f"Ingekort  : {len(test_tekst)} tekens")
print(f"Criterium : I-An 2.1\n")

# Sla tussenresultaten op
resultaten_test = {}

# === VARIANT A: Zero-shot ===
print("="*50)
print("VARIANT A — Zero-shot")
print("="*50)
prompt_zs = bouw_prompt_zero_shot("I-An 2.1", test_tekst)
print(f"Promptlengte: {len(prompt_zs)} tekens | max output: 300 tokens\n")
resultaat_zs = ollama_aanroep(prompt_zs, max_tokens=300, stream=True)
resultaten_test["zero_shot"] = resultaat_zs

# === VARIANT B: Few-shot ===
print("\n" + "="*50)
print("VARIANT B — Few-shot")
print("="*50)
prompt_fs = bouw_prompt_few_shot("I-An 2.1", test_tekst)
print(f"Promptlengte: {len(prompt_fs)} tekens | max output: 300 tokens\n")
resultaat_fs = ollama_aanroep(prompt_fs, max_tokens=300, stream=True)
resultaten_test["few_shot"] = resultaat_fs

Document  : GroepB11_PvA_Tussentijds_Goed_2.pdf
Origineel : 18118 tekens
Ingekort  : 2520 tekens
Criterium : I-An 2.1

VARIANT A — Zero-shot
Promptlengte: 4548 tekens | max output: 300 tokens

Model reageert (streaming):
----------------------------------------
FEEDUP (wat was het doel):
Het doel is om een methodisch onderzoek en analyse uit te voeren in de context van infrastructuur voor een datavraagstuk, met aansluiting op de gegeven opdracht.

FEEDBACK (hoe staat het ervoor):
De student heeft een aantal belangrijke aspecten aanrisico's en oplossingsrichtingen besproken. Het document bevat ook een ethische kader, wat een goed begin is om te beschouwen in methodisch onderzoek. Hier zijn echter enkele gebieden waar de student kan verbeteren:

1. **Context**: De context van het project wordt niet volledig uitgelegd. Het zou nuttig zijn om meer informatie te geven over de huidige situatie, bedrijfsprocesen en de gewenste toekomstige situatie.
2. **Probleemstelling**: Hoewel er risico's 

### 3.5 — Resultaten: vergelijking van promptvarianten

**Testconfiguratie:**
- Document: GroepB11_PvA_Tussentijds_Goed_2.pdf (ingekort naar 2520 tekens)
- Criterium: I-An 2.1 — Methodisch onderzoek en analyse
- Model: Qwen 2.5 7B | Temperatuur: 0.3 | Max tokens: 300

**Observaties:**

| Aspect | Variant A (Zero-shot) | Variant B (Few-shot) |
|---|---|---|
| Structuur (feedup/feedback/feedforward) |  Gevolgd | Gevolgd |
| Responstijd | 720s (~12 min) | 945s (~16 min) |
| Responselengte | 1071 tekens | 1107 tekens |
| Specifieke verbeterpunten | Context, probleemstelling, requirementsanalyse | Probleemstelling, requirementsanalyse, scope |
| Rubric-aansluiting | Gedeeltelijk — noemt vereiste onderdelen maar mist verwijzing naar beoordelingscriteria | Beter — koppelt expliciet aan technische vraagstukken en requirementsanalyse |
| Taalfouten | "aanrisico's", "ontbreend" | Geen opvallende fouten |
| Toon | Constructief | Constructief, iets gerichter |

**Eerste conclusie:**
- Beide varianten volgen de gevraagde structuur correct.
- De few-shot variant levert meer rubric-gerichte feedback op en maakt minder taalfouten, maar is ~30% langzamer door de langere prompt.
- De output wordt afgekapt bij 300 tokens — in de volledige pipeline verhogen we dit zodat de feedforward-sectie compleet kan worden afgemaakt.
- De verwerkingstijd (~12–16 min per aanroep) is een beperking voor de schaalbaarheid. Dit wordt meegenomen in de evaluatie bij DV3.

---
## Sectie 4 — DV3: Modelvergelijking (A/B-test)

**Deelvraag 3:** Welke open-source LLM is het meest geschikt voor het genereren van rubric-gebaseerde formatieve feedback op studentopleveringen?

In deze sectie voeren we een A/B-test uit met minimaal twee lokaal beschikbare modellen via Ollama. Beide modellen verwerken exact dezelfde geanonimiseerde studentopleveringen met identieke prompts. We vergelijken de output op:

- **Feedbackkwaliteit** — volgt de output de feedup/feedback/feedforward-structuur?
- **Rubric-aansluiting** — verwijst de feedback naar de beoordelingscriteria?
- **Nederlandse taalvaardigheid** — grammatica, spelling, vloeiendheid
- **Verwerkingstijd** — hoe lang duurt een aanroep?
- **Consistentie** — geeft het model vergelijkbare feedback bij vergelijkbare documenten?

De resultaten worden vastgelegd in een overzichtstabel.

### 4.1 — Modelselectie en motivatie

We selecteren twee modellen die lokaal beschikbaar zijn via Ollama en geschikt zijn voor Nederlandstalige tekstgeneratie. Beide modellen worden gecontroleerd op beschikbaarheid.

In [36]:
# Cel 4.1 — Modelselectie en beschikbaarheidscheck

MODELLEN = {
    "qwen2.5:7b": {
        "naam": "Qwen 2.5 7B",
        "reden": (
            "Qwen 2.5 is getraind op meertalige data waaronder Nederlands. "
            "De 7B-variant is het hoofdmodel voor deze POC en heeft in de "
            "eerste tests bruikbare formatieve feedback gegenereerd."
        )
    },
    "qwen2.5:3b": {
        "naam": "Qwen 2.5 3B",
        "reden": (
            "De 3B-variant uit dezelfde Qwen 2.5-familie is aanzienlijk sneller "
            "op lokale hardware. Door een groter en kleiner model uit dezelfde "
            "familie te vergelijken wordt zichtbaar welke impact modelgrootte "
            "heeft op de kwaliteit van formatieve feedback."
        )
    }
}

# Check beschikbaarheid
beschikbare_modellen = []

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    if response.status_code == 200:
        geinstalleerd = [m["name"] for m in response.json().get("models", [])]
        print(f"Geïnstalleerde modellen: {geinstalleerd}\n")

        for model_id, info in MODELLEN.items():
            gevonden = any(model_id in naam for naam in geinstalleerd)
            status = " Beschikbaar" if gevonden else " Niet geïnstalleerd"
            print(f"  {info['naam']} ({model_id}): {status}")
            if gevonden:
                beschikbare_modellen.append(model_id)

        if len(beschikbare_modellen) < 2:
            ontbrekend = [m for m in MODELLEN if m not in beschikbare_modellen]
            print(f"\n  Installeer: ollama pull {ontbrekend[0]}")
except requests.ConnectionError:
    print("FOUT: Ollama niet bereikbaar")

Geïnstalleerde modellen: ['qwen2.5:3b', 'mistral:latest', 'qwen2.5:7b', 'llama3.1:latest']

  Qwen 2.5 7B (qwen2.5:7b):  Beschikbaar
  Qwen 2.5 3B (qwen2.5:3b):  Beschikbaar


### 4.2 — A/B-test opzet

We draaien de test per document in aparte cellen, zodat tussenresultaten bewaard blijven als een aanroep lang duurt of faalt. Resultaten worden na elke aanroep opgeslagen naar JSON.

In [37]:
# Cel 4.2a — A/B-test opzet en resultaten-opslag

# Laad eventueel eerder opgeslagen resultaten
ab_pad = os.path.join(OUTPUT_DIR, "ab_test_resultaten.json")

if os.path.exists(ab_pad):
    with open(ab_pad, "r", encoding="utf-8") as f:
        ab_resultaten = json.load(f)
    print(f"Bestaande resultaten geladen: {ab_pad}")
else:
    ab_resultaten = {}
    print("Nieuwe test gestart.")

def sla_resultaten_op():
    """Slaat de huidige resultaten op na elke aanroep."""
    with open(ab_pad, "w", encoding="utf-8") as f:
        json.dump(ab_resultaten, f, ensure_ascii=False, indent=2)
    print(f"Resultaten opgeslagen: {ab_pad}")

# Bereid de testdocumenten voor
test_documenten = {}
for groep in ["B11", "C8", "A2"]:
    doc = verwerkte_documenten[groep][0]
    test_documenten[groep] = {
        "bestandsnaam": doc["bestandsnaam"],
        "tekst": extraheer_relevante_secties(
            doc["geanonimiseerde_tekst"],
            max_tekens=2500
        )
    }
    print(f"Groep {groep}: {doc['bestandsnaam']} ({len(test_documenten[groep]['tekst'])} tekens)")

Bestaande resultaten geladen: output/ab_test_resultaten.json
Groep B11: GroepB11_PvA_Tussentijds_Goed_2.pdf (2520 tekens)
Groep C8: GroepC8_PvA_Tussentijds_Voldoende_2.pdf (2520 tekens)
Groep A2: GroepA2_PvA_Tussentijds_Onvoldoende_1.pdf (2520 tekens)


In [38]:
# Cel 4.2e — Qwen 2.5 3B × Groep B11

if "qwen2.5:3b" not in ab_resultaten:
    ab_resultaten["qwen2.5:3b"] = {}

if "B11" not in ab_resultaten.get("qwen2.5:3b", {}):
    prompt = bouw_prompt_few_shot("I-An 2.1", test_documenten["B11"]["tekst"])
    resultaat = ollama_aanroep(prompt, model="qwen2.5:3b", max_tokens=500, stream=True)
    if resultaat:
        ab_resultaten["qwen2.5:3b"]["B11"] = resultaat
        sla_resultaten_op()
else:
    print("Groep B11 × Qwen 3B: al aanwezig in resultaten ✅")
    print(f"Duur: {ab_resultaten['qwen2.5:3b']['B11']['duur_seconden']}s")

Model reageert (streaming):
----------------------------------------
**FEEDUP (wat was het doel):**

Bij criterium I-An 2.1 wordt verwacht dat het Plan van Aanpak een methodische analyse bevat van het data-infrastructuurprobleem. Het document moet een relevante en afgebakende probleemstelling bevatten, plus een analyse van de organisatie, betrokkenen en omgevingsfactoren, verwerkt in een requirementsanalyse.

**FEEDBACK (hoe staat het ervoor):**

De context is helder beschreven en geeft goed weer in welke setting het project plaatsvindt. De probleemstelling is aanwezig maar nog vrij breed geformuleerd — het is niet duidelijk welk specifiek data-infrastructuurprobleem wordt onderzocht. Het document bevat een risicolijst met relevante risico's en mitigaties, maar de koppeling met de probleemstelling ontbreekt.

De requirementsanalyse bevat functionele en niet-functionele requirements, maar de koppeling met de probleemstelling ontbreekt. De stakeholderanalyse is nog niet aanwezig in het d

In [39]:
# Cel 4.2f — Qwen 2.5 3B × Groep C8

if "C8" not in ab_resultaten.get("qwen2.5:3b", {}):
    prompt = bouw_prompt_few_shot("I-An 2.1", test_documenten["C8"]["tekst"])
    resultaat = ollama_aanroep(prompt, model="qwen2.5:3b", max_tokens=500, stream=True)
    if resultaat:
        ab_resultaten["qwen2.5:3b"]["C8"] = resultaat
        sla_resultaten_op()
else:
    print("Groep C8 × Qwen 3B: al aanwezig in resultaten ✅")
    print(f"Duur: {ab_resultaten['qwen2.5:3b']['C8']['duur_seconden']}s")

Model reageert (streaming):
----------------------------------------
FEEDUP (wat was het doel):
Bij criterium I-An 2.1 wordt verwacht dat het Plan van Aanpak een methodische analyse bevat van het data-infrastructuurprobleem. Het document moet een relevante en afgebakende probleemstelling bevatten, plus een analytische beschrijving van de betrokken organisatie(s) en omgevingsfactoren, verwerkt in een requirementsanalyse.

FEEDBACK (hoe staat het ervoor):
De context is helder beschreven en geeft goed weer in welke setting het project plaatsvindt. De probleemstelling is aanwezig maar nog vrij breed geformuleerd — het is niet duidelijk welk specifiek data-infrastructuurprobleem wordt onderzocht. Het BPMN-schema van de huidige en gewenste operatieproces heeft een interessante structuur, maar de betrokken organisaties (bijv. personeel in OK-Planning, operatieassistenten, chirurg, specialist) zijn niet detaillerd genoemd of geanalyseerd.

De requirementsanalyse bevat functionele en niet-funct

In [40]:
# Cel 4.2g — Qwen 2.5 3B × Groep A2

if "A2" not in ab_resultaten.get("qwen2.5:3b", {}):
    prompt = bouw_prompt_few_shot("I-An 2.1", test_documenten["A2"]["tekst"])
    resultaat = ollama_aanroep(prompt, model="qwen2.5:3b", max_tokens=500, stream=True)
    if resultaat:
        ab_resultaten["qwen2.5:3b"]["A2"] = resultaat
        sla_resultaten_op()
else:
    print("Groep A2 × Qwen 3B: al aanwezig in resultaten ✅")
    print(f"Duur: {ab_resultaten['qwen2.5:3b']['A2']['duur_seconden']}s")

Model reageert (streaming):
----------------------------------------
**FEEDUP (wat was het doel):**

Bij criterium I-An 2.1 wordt verwacht dat het Plan van Aanpak een methodische analyse bevat van het data-infrastructuurprobleem. Het document moet een relevante en afgebakende probleemstelling bevatten, plus een analyse van de organisatie, betrokkenen en omgevingsfactoren, verwerkt in een requirementsanalyse.

**FEEDBACK (hoe staat het ervoor):**

De context is helder beschreven en geeft goed weer in welke setting het project plaatsvindt. De probleemstelling is aanwezig maar nog vrij breed geformuleerd — het is niet duidelijk welk specifiek data-infrastructuurprobleem wordt onderzocht. De requirementsanalyse bevat functionele en niet-functionele requirements, maar de koppeling met de probleemstelling ontbreekt. De risicolijst is aanwezig en bevat relevante risico's met mitigaties.

**FEEDFORWARD (wat is de volgende stap):**

1. Verscherp de probleemstelling: formuleer één concrete hoofd

### 4.6 — Analyse en conclusie DV3

**Kwantitatieve vergelijking:**

| Metriek | Qwen 2.5 7B | Qwen 2.5 3B |
|---|---|---|
| Gem. responstijd | 2701s (~45 min) | 452s (~7.5 min) |
| Gem. responselengte | 1775 tekens | 1610 tekens |
| Structuur (feedup/feedback/feedforward) | 3/3 | 3/3 |
| Snelheidsfactor | 1× (baseline) | ~6× sneller |

**Kwalitatieve observaties:**

- **Structuuropvolging:** beide modellen volgen de feedup/feedback/feedforward-structuur correct in alle drie de documenten.
- **Rubric-aansluiting:** beide modellen verwijzen naar de vereiste onderdelen (probleemstelling, requirementsanalyse, risicolijst, scope). Het 7B-model gaat iets dieper in op documentspecifieke details.
- **Specificiteit:** het 7B-model noemt vaker concrete elementen uit het document (bijv. "baliepersoneel en patienten", "BPMN-schema", "GGZ Rivierduinen"). Het 3B-model is iets generieker maar benoemt ook documentspecifieke onderdelen (bijv. "HiX EPD-systeem", "OK-Planning").
- **Taalvaardigheid:** het 7B-model maakt af en toe spelfouten ("inclusief", "ontbreend"). Het 3B-model schrijft over het algemeen vloeiender Nederlands.
- **Consistentie:** het 3B-model geeft vrij vergelijkbare feedback op alle drie de documenten (vergelijkbare structuur en diepgang). Het 7B-model varieert meer in aanpak per document.
- **Feedforward-kwaliteit:** beide modellen geven concrete verbeterpunten. Het 7B-model geeft soms voorbeeldzinnen, het 3B-model houdt het bij richtlijnen.

**Conclusie DV3:**

Het 3B-model is voor deze use case het meest praktisch: het levert vergelijkbare feedbackkwaliteit bij ~6× hogere snelheid. Het 7B-model genereert iets meer documentspecifieke details, maar het verschil rechtvaardigt niet de factor 6 in verwerkingstijd — zeker niet op hardware zonder GPU-acceleratie. Voor een productie-implementatie op lokale hardware is Qwen 2.5 3B het aanbevolen model. Het 7B-model kan worden ingezet wanneer betere hardware beschikbaar is.

---
## Sectie 5 — DV4: Vergelijking met Gouden Standaard

**Deelvraag 4:** In welke mate sluit de door de LLM gegenereerde feedback aan op de kenmerken van effectieve formatieve feedback zoals vastgesteld in deelvraag 1?

In deze sectie vergelijken we de LLM-output met de docentfeedback uit `Formatieve_Feedback_Gecombineerd_Alle_Groepen.pdf`. Op basis van het evaluatiekader uit DV1 scoren we beide op de zeven dimensies (feedup, feedback, feedforward, specificiteit, rubric-aansluiting, constructieve toon, consistentie). De analyse richt zich op de vraag of de LLM-feedback inhoudelijk vergelijkbaar is met de referentiefeedback van de docent.

### 5.1 — Docentfeedback laden en structureren

We extraheren de docentfeedback uit de PDF en structureren deze per groep. We matchen de feedback op de juiste documentversie: de LLM heeft de tussentijdse versies verwerkt, dus we gebruiken feedbackronde 1 (op v1.0) als referentie.

In [41]:
# Cel 5.1 — Docentfeedback laden en structureren per groep

import pdfplumber

FEEDBACK_PAD = "/Users/user/Desktop/POC/INDATAD/Feedback docent/Formatieve_Feedback_Gecombineerd_Alle_Groepen.pdf"

with pdfplumber.open(FEEDBACK_PAD) as pdf:
    volledige_tekst = ""
    for p in pdf.pages:
        t = p.extract_text()
        if t:
            volledige_tekst += t + "\n\n"

# Splits de feedback per groep op basis van de koppen
import re

secties = re.split(r'FORMATIEVE FEEDBACK\n', volledige_tekst)

docent_feedback = {}

# Zoek per groep de relevante feedback
for sectie in secties:
    if "Groep B11" in sectie[:200]:
        docent_feedback["B11"] = sectie
    elif "Groep C8" in sectie[:200]:
        docent_feedback["C8"] = sectie
    elif "Groep A2" in sectie[:200]:
        docent_feedback["A2"] = sectie

# Sla op als JSON voor hergebruik
docent_pad = os.path.join(OUTPUT_DIR, "docent_feedback_per_groep.json")
with open(docent_pad, "w", encoding="utf-8") as f:
    json.dump(docent_feedback, f, ensure_ascii=False, indent=2)

for groep, tekst in docent_feedback.items():
    print(f"Groep {groep}: {len(tekst)} tekens geladen")
    # Toon eerste 200 tekens als check
    print(f"  Begin: {tekst[:150].strip()}...")
    print()

print(f"Opgeslagen: {docent_pad}")

Groep B11: 6215 tekens geladen
  Begin: Plan van Aanpak – Twee feedbackrondes
Groep B11 | Module INDATAD Data Advanced | Franciscus Ziekenhuis
Opdrachtgever: Sandra Mohan-Khemon – Franciscus...

Groep C8: 5718 tekens geladen
  Begin: Plan van Aanpak – Twee feedbackrondes
Groep C8 | Module INDATAD Data Advanced | Haaglanden Medisch Centrum
Opdrachtgever: Xandra Riswick – Haaglanden...

Groep A2: 6807 tekens geladen
  Begin: Plan van Aanpak – Twee feedbackrondes
Groep A2 | Module INDATAD Data Advanced | GGZ Rivierduinen
Opdrachtgever: Jasper Duijndam – GGZ Rivierduinen
Stu...

Opgeslagen: output/docent_feedback_per_groep.json


### 5.2 — Zij-aan-zij vergelijking: docent vs. LLM

Per groep tonen we de docentfeedback (ronde 1) naast de LLM-feedback (Qwen 2.5 7B, few-shot). Zo wordt direct zichtbaar waar de LLM overeenkomt met de docent en waar ze afwijken.

In [42]:
# Cel 5.2 — Zij-aan-zij vergelijking per groep

for groep in ["B11", "C8", "A2"]:
    print(f"\n{'#'*70}")
    print(f"GROEP {groep}")
    print(f"{'#'*70}")

    # Docentfeedback: toon eerste feedbackronde (op v1.0)
    docent = docent_feedback.get(groep, "")
    # Pak alleen feedbackronde 1
    ronde1_eind = docent.find("Feedbackronde 2")
    if ronde1_eind > 0:
        docent_ronde1 = docent[:ronde1_eind].strip()
    else:
        docent_ronde1 = docent.strip()

    print(f"\n--- DOCENT (feedbackronde 1) ---")
    print(docent_ronde1[:1500])
    if len(docent_ronde1) > 1500:
        print(f"\n[... nog {len(docent_ronde1) - 1500} tekens ...]")

    # LLM-feedback
    llm = ab_resultaten.get("qwen2.5:7b", {}).get(groep, {})
    llm_tekst = llm.get("response", "(geen output)")

    print(f"\n--- LLM: Qwen 2.5 7B ---")
    print(llm_tekst)
    print()


######################################################################
GROEP B11
######################################################################

--- DOCENT (feedbackronde 1) ---
Plan van Aanpak – Twee feedbackrondes
Groep B11 | Module INDATAD Data Advanced | Franciscus Ziekenhuis
Opdrachtgever: Sandra Mohan-Khemon – Franciscus Gasthuis & Vlietland
Studenten: Artem Stasyuk, Lukas Ooms, Noa de Lange, Sara Denno,
Tobias van Muiswinkel
Feedbackronde 1: Feedback op versie 1 (v1.0)
1. Samenvatting
Het plan van aanpak van groep B11 behandelt de hoge werkdruk van het baliepersoneel op
de MDL-afdeling van het Franciscus Ziekenhuis. Het document bevat alle verplichte
rubriconderdelen en is op de meeste punten inhoudelijk sterk uitgewerkt. De context
beschrijft zowel maatschappelijke als technologische ontwikkelingen met concrete
verwijzingen naar HiX en de BeterDichtbij-app. De organisatiebeschrijving bevat een
organogram, relevante afdelingen, IT-systemen en een specifieke koppeling na

### 5.3 — Scoring op het evaluatiekader

We scoren zowel de docentfeedback als de LLM-feedback per groep op de zeven dimensies uit het evaluatiekader (DV1). De scores worden handmatig toegekend op basis van de output hierboven en vastgelegd in een dictionary. Dit maakt de vergelijking reproduceerbaar.

In [43]:
# Cel 5.3 — Scoring: docent vs. LLM op de zeven dimensies

# Scores worden handmatig toegekend na inspectie van cel 5.2
# Schaal: 1 = onvoldoende, 2 = gedeeltelijk, 3 = volledig

scores_vergelijking = {
    "B11": {
        "docent": {
            "feedup": 3,       # Docent benoemt expliciet de rubric en verwachtingen
            "feedback": 3,     # Gedetailleerde beschrijving met verwijzing naar specifieke onderdelen
            "feedforward": 3,  # Concrete verbeterpunten (Gantt chart, bibliografie uitbreiden)
            "specificiteit": 3, # Verwijst naar HiX, BeterDichtbij-app, BPMN, MoSCoW
            "rubric_aansluiting": 3, # Dekt alle rubric-onderdelen systematisch
            "constructieve_toon": 3, # Balans tussen tops en tips
            "consistentie": 3       # Zelfde structuur als bij andere groepen
        },
        "llm_7b": {
            "feedup": 2,       # Noemt het doel maar niet de specifieke rubric-verwachtingen
            "feedback": 2,     # Bespreekt onderdelen maar minder specifiek dan docent
            "feedforward": 2,  # Geeft verbeterpunten maar soms generiek
            "specificiteit": 2, # Noemt "baliepersoneel" maar mist HiX, BPMN etc.
            "rubric_aansluiting": 2, # Noemt vereiste onderdelen maar mist sommige
            "constructieve_toon": 3, # Toon is constructief en respectvol
            "consistentie": 2       # Te beoordelen in combinatie met andere groepen
        }
    },
    "C8": {
        "docent": {
            "feedup": 3,
            "feedback": 3,
            "feedforward": 3,
            "specificiteit": 3,
            "rubric_aansluiting": 3,
            "constructieve_toon": 3,
            "consistentie": 3
        },
        "llm_7b": {
            "feedup": 2,
            "feedback": 2,
            "feedforward": 2,
            "specificiteit": 2,
            "rubric_aansluiting": 2,
            "constructieve_toon": 3,
            "consistentie": 2
        }
    },
    "A2": {
        "docent": {
            "feedup": 3,
            "feedback": 3,
            "feedforward": 3,
            "specificiteit": 3,
            "rubric_aansluiting": 3,
            "constructieve_toon": 3,
            "consistentie": 3
        },
        "llm_7b": {
            "feedup": 2,
            "feedback": 3,
            "feedforward": 2,
            "specificiteit": 2,
            "rubric_aansluiting": 2,
            "constructieve_toon": 3,
            "consistentie": 2
        }
    }
}

# Opslaan
scores_pad = os.path.join(OUTPUT_DIR, "scores_vergelijking.json")
with open(scores_pad, "w", encoding="utf-8") as f:
    json.dump(scores_vergelijking, f, ensure_ascii=False, indent=2)
print(f"Scores opgeslagen: {scores_pad}")

Scores opgeslagen: output/scores_vergelijking.json


### 5.4 — Overzichtstabel en analyse

We visualiseren de scores in een overzichtstabel: docent vs. LLM, per dimensie, gemiddeld over alle drie de groepen.

In [44]:
# Cel 5.4 — Overzichtstabel: docent vs. LLM per dimensie

dimensies = list(EVALUATIEKADER.keys())

# Bereken gemiddelden over alle groepen
gem_docent = {}
gem_llm = {}

for dim in dimensies:
    docent_scores = [scores_vergelijking[g]["docent"][dim] for g in ["B11", "C8", "A2"]]
    llm_scores = [scores_vergelijking[g]["llm_7b"][dim] for g in ["B11", "C8", "A2"]]
    gem_docent[dim] = sum(docent_scores) / len(docent_scores)
    gem_llm[dim] = sum(llm_scores) / len(llm_scores)

print(f"{'='*65}")
print(f"VERGELIJKING DOCENT vs. LLM (Qwen 2.5 7B)")
print(f"Gemiddelde scores over 3 groepen — schaal 1 (laag) tot 3 (hoog)")
print(f"{'='*65}")
print(f"\n{'Dimensie':<25} {'Docent':<12} {'LLM 7B':<12} {'Verschil':<12}")
print("-" * 60)

totaal_docent = 0
totaal_llm = 0

for dim in dimensies:
    d = gem_docent[dim]
    l = gem_llm[dim]
    verschil = l - d
    teken = "+" if verschil > 0 else ""
    totaal_docent += d
    totaal_llm += l
    print(f"{dim:<25} {d:<12.1f} {l:<12.1f} {teken}{verschil:<12.1f}")

print("-" * 60)
gem_d = totaal_docent / len(dimensies)
gem_l = totaal_llm / len(dimensies)
print(f"{'GEMIDDELD':<25} {gem_d:<12.1f} {gem_l:<12.1f} {gem_l - gem_d:+.1f}")
print(f"\nDocent totaal: {totaal_docent:.0f}/21 | LLM totaal: {totaal_llm:.0f}/21")

VERGELIJKING DOCENT vs. LLM (Qwen 2.5 7B)
Gemiddelde scores over 3 groepen — schaal 1 (laag) tot 3 (hoog)

Dimensie                  Docent       LLM 7B       Verschil    
------------------------------------------------------------
feedup                    3.0          2.0          -1.0        
feedback                  3.0          2.3          -0.7        
feedforward               3.0          2.0          -1.0        
specificiteit             3.0          2.0          -1.0        
rubric_aansluiting        3.0          2.0          -1.0        
constructieve_toon        3.0          3.0          0.0         
consistentie              3.0          2.0          -1.0        
------------------------------------------------------------
GEMIDDELD                 3.0          2.2          -0.8

Docent totaal: 21/21 | LLM totaal: 15/21


---
## Sectie 6 — DV5: Guardrails

**Deelvraag 5:** Hoe kunnen rolgebonden toegang, transparantie en logging als guardrails worden geïmplementeerd binnen een lokaal gehoste LLM-omgeving voor formatieve feedback?

In deze sectie implementeren we drie guardrails op POC-niveau:

1. **Rolgebonden toegang** — een docent ziet de volledige feedback inclusief modelinstellingen en logs. Een student ziet alleen de formatieve feedback zonder technische details.
2. **Transparantie** — bij elke output wordt gelogd welk model, welke temperatuur, welke prompt en welke versie actief waren.
3. **Logging** — alle invoer en uitvoer wordt bewaard in een lokaal logbestand (JSONL-formaat).

Na implementatie testen we of de guardrails werken en in hoeverre ze aansluiten op principes van privacy, reproduceerbaarheid en controleerbaarheid.

### 6.1 — Rolgebonden toegang

We implementeren een eenvoudig rolsysteem met twee rollen: `docent` en `student`. De rol bepaalt welke informatie wordt getoond. Een docent ziet alles; een student ziet alleen de formatieve feedback.

In [45]:
# Cel 6.1 — Rolgebonden toegang

ROLLEN = {
    "docent": {
        "beschrijving": "Volledige toegang tot feedback, modelinstellingen, logs en prompts",
        "ziet_feedback": True,
        "ziet_modelinstellingen": True,
        "ziet_prompt": True,
        "ziet_logs": True,
        "ziet_scores": True
    },
    "student": {
        "beschrijving": "Alleen formatieve feedback, geen technische details",
        "ziet_feedback": True,
        "ziet_modelinstellingen": False,
        "ziet_prompt": False,
        "ziet_logs": False,
        "ziet_scores": False
    }
}

def toon_feedback(resultaat, rol="student"):
    """
    Toont feedback op basis van de rol van de gebruiker.

    Parameters:
        resultaat : dict uit ollama_aanroep() of ab_resultaten
        rol       : "docent" of "student"
    """
    if rol not in ROLLEN:
        print(f"Onbekende rol: '{rol}'. Kies 'docent' of 'student'.")
        return

    rechten = ROLLEN[rol]
    print(f"[Weergave voor: {rol.upper()}]")
    print("=" * 50)

    # Feedback is altijd zichtbaar
    if rechten["ziet_feedback"]:
        print("\nFORMATIEVE FEEDBACK:")
        print("-" * 40)
        print(resultaat.get("response", "(geen feedback beschikbaar)"))

    # Modelinstellingen: alleen voor docent
    if rechten["ziet_modelinstellingen"]:
        print(f"\n--- MODELINSTELLINGEN (alleen zichtbaar voor docent) ---")
        print(f"  Model: {resultaat.get('model', '?')}")
        print(f"  Temperatuur: {resultaat.get('temperatuur', '?')}")
        print(f"  Responstijd: {resultaat.get('duur_seconden', '?')}s")
        print(f"  Promptlengte: {resultaat.get('prompt_lengte', '?')} tekens")
        print(f"  Responselengte: {resultaat.get('response_lengte', '?')} tekens")
        print(f"  Timestamp: {resultaat.get('timestamp', '?')}")

    # Prompt: alleen voor docent
    if rechten["ziet_prompt"] and "prompt" in resultaat:
        print(f"\n--- GEBRUIKTE PROMPT (alleen zichtbaar voor docent) ---")
        prompt = resultaat.get("prompt", "")
        if len(prompt) > 500:
            print(prompt[:500] + "\n[... ingekort ...]")
        else:
            print(prompt)

    print()


# Test met een echt resultaat
test_res = ab_resultaten["qwen2.5:7b"]["B11"]

print(">>> STUDENTWEERGAVE:\n")
toon_feedback(test_res, rol="student")

print("\n>>> DOCENTWEERGAVE:\n")
toon_feedback(test_res, rol="docent")

>>> STUDENTWEERGAVE:

[Weergave voor: STUDENT]

FORMATIEVE FEEDBACK:
----------------------------------------
FEEDUP (wat was het doel):
Het doel is om een methodisch onderzoek en analyse uit te voeren op het gebied van infrastructuur voor een datavraagstuk, met aansluiting op de gegeven context.

FEEDBACK (hoe staat het ervoor):
De student heeft een duidelijke risicolijst opgesteld, inclusief mogelijke maatregelen. De ethische principes zijn goed beschreven en koppelen aan de opdracht. Het document toont een goede intentie om methodisch te werken, maar er zijn enkele tekortkomingen die verbeterd kunnen worden.

1. **Context**: Er is een duidelijke context gegeven, inclusief het baliepersoneel en patienten.
2. **Probleemstelling**: De ethische principes vormen de basis van de problematiek, maar er is geen specifieke data-infrastructuurprobleem benoemd. Het blijft een algemeen kader zonder concreet vraagstuk.
3. **Requirementsanalyse**: Er zijn ethische aspecten besproken, maar deze val

### 6.2 — Transparantie: metadata bij elke output

We maken een functie die een transparantierapport genereert voor elke feedback-output. Dit rapport bevat alle informatie die nodig is om de output te reproduceren: model, versie, temperatuur, prompt, inputlengte en timestamp.

In [46]:
# Cel 6.2 — Transparantierapport per feedback-output

def genereer_transparantierapport(resultaat, groep, criterium, prompt_variant):
    """
    Genereert een transparantierapport dat bij elke output hoort.
    Maakt de omstandigheden van de feedback volledig reproduceerbaar.
    """
    rapport = {
        "groep": groep,
        "criterium": criterium,
        "prompt_variant": prompt_variant,
        "model": resultaat.get("model", "onbekend"),
        "temperatuur": resultaat.get("temperatuur", "onbekend"),
        "max_tokens": 500,
        "prompt_lengte_tekens": resultaat.get("prompt_lengte", 0),
        "response_lengte_tekens": resultaat.get("response_lengte", 0),
        "verwerkingstijd_seconden": resultaat.get("duur_seconden", 0),
        "timestamp": resultaat.get("timestamp", "onbekend"),
        "input_tekst_max_tekens": 2500,
        "ollama_url": OLLAMA_BASE_URL,
        "anonimisering_toegepast": True,
        "afbeeldingen_verwerkt": False,
        "afbeeldingen_gelogd": True
    }
    return rapport


# Genereer rapporten voor alle bestaande resultaten
transparantie_rapporten = []

for model_id in ab_resultaten:
    for groep in ab_resultaten[model_id]:
        res = ab_resultaten[model_id][groep]
        rapport = genereer_transparantierapport(
            resultaat=res,
            groep=groep,
            criterium="I-An 2.1",
            prompt_variant="few-shot"
        )
        transparantie_rapporten.append(rapport)

# Opslaan
rapport_pad = os.path.join(LOG_DIR, "transparantie_rapporten.json")
with open(rapport_pad, "w", encoding="utf-8") as f:
    json.dump(transparantie_rapporten, f, ensure_ascii=False, indent=2)

print(f"Transparantierapporten opgeslagen: {rapport_pad}")
print(f"Totaal: {len(transparantie_rapporten)} rapporten\n")

# Toon een voorbeeld
print("Voorbeeld transparantierapport:")
print("-" * 40)
for k, v in transparantie_rapporten[0].items():
    print(f"  {k}: {v}")

Transparantierapporten opgeslagen: logs/transparantie_rapporten.json
Totaal: 6 rapporten

Voorbeeld transparantierapport:
----------------------------------------
  groep: B11
  criterium: I-An 2.1
  prompt_variant: few-shot
  model: qwen2.5:7b
  temperatuur: 0.3
  max_tokens: 500
  prompt_lengte_tekens: 5990
  response_lengte_tekens: 1786
  verwerkingstijd_seconden: 3827.85
  timestamp: 2026-04-02T17:36:54.194918
  input_tekst_max_tekens: 2500
  ollama_url: http://localhost:11434
  anonimisering_toegepast: True
  afbeeldingen_verwerkt: False
  afbeeldingen_gelogd: True


### 6.3 — Logging: overzicht van alle logs

We geven een overzicht van alle logbestanden die gedurende de notebook zijn aangemaakt. Dit maakt inzichtelijk waar alle invoer, uitvoer en metadata zijn opgeslagen.

In [48]:
# Cel 6.3 — Overzicht van alle logbestanden

print("OVERZICHT LOGBESTANDEN")
print("=" * 60)

logbestanden = {
    "Ollama-aanroepen": os.path.join(LOG_DIR, "ollama_aanroepen.jsonl"),
    "Afbeelding-log": os.path.join(LOG_DIR, "afbeelding_log.json"),
    "Transparantierapporten": os.path.join(LOG_DIR, "transparantie_rapporten.json"),
}

output_bestanden = {
    "A/B-test resultaten": os.path.join(OUTPUT_DIR, "ab_test_resultaten.json"),
    "Geanonimiseerde documenten": os.path.join(OUTPUT_DIR, "geanonimiseerd", "verwerkte_documenten.json"),
    "Docent feedback": os.path.join(OUTPUT_DIR, "docent_feedback_per_groep.json"),
    "Scores vergelijking": os.path.join(OUTPUT_DIR, "scores_vergelijking.json"),
}

print("\nLOGS:")
for naam, pad in logbestanden.items():
    if os.path.exists(pad):
        grootte = os.path.getsize(pad)
        print(f"   {naam}: {pad} ({grootte} bytes)")
    else:
        print(f"   {naam}: {pad} (niet gevonden)")

print("\nOUTPUT:")
for naam, pad in output_bestanden.items():
    if os.path.exists(pad):
        grootte = os.path.getsize(pad)
        print(f"  {naam}: {pad} ({grootte} bytes)")
    else:
        print(f"   {naam}: {pad} (niet gevonden)")

# Tel totaal Ollama-aanroepen
log_pad = os.path.join(LOG_DIR, "ollama_aanroepen.jsonl")
if os.path.exists(log_pad):
    with open(log_pad, "r") as f:
        aantal = len(f.readlines())
    print(f"\nTotaal gelogde Ollama-aanroepen: {aantal}")

OVERZICHT LOGBESTANDEN

LOGS:
   Ollama-aanroepen: logs/ollama_aanroepen.jsonl (18133 bytes)
   Afbeelding-log: logs/afbeelding_log.json (5053 bytes)
   Transparantierapporten: logs/transparantie_rapporten.json (3048 bytes)

OUTPUT:
  A/B-test resultaten: output/ab_test_resultaten.json (11727 bytes)
  Geanonimiseerde documenten: output/geanonimiseerd/verwerkte_documenten.json (284605 bytes)
  Docent feedback: output/docent_feedback_per_groep.json (19120 bytes)
  Scores vergelijking: output/scores_vergelijking.json (1239 bytes)

Totaal gelogde Ollama-aanroepen: 13


### 6.4 — Test en conclusie DV5

We testen of de drie guardrails correct werken en concluderen in hoeverre ze aansluiten op principes van privacy, reproduceerbaarheid en controleerbaarheid.

In [49]:
# Cel 6.4 — Test en conclusie DV5

print("TEST GUARDRAILS")
print("=" * 60)

# Test 1: Rolgebonden toegang
print("\n1. ROLGEBONDEN TOEGANG")
print("-" * 40)
student_velden = [k for k, v in ROLLEN["student"].items() if v is True and k != "beschrijving"]
docent_velden = [k for k, v in ROLLEN["docent"].items() if v is True and k != "beschrijving"]
print(f"  Student ziet: {', '.join(student_velden)}")
print(f"  Docent ziet:  {', '.join(docent_velden)}")
print(f"   Rolgebonden toegang werkt — student ziet {len(student_velden)} velden, "
      f"docent ziet {len(docent_velden)} velden")

# Test 2: Transparantie
print("\n2. TRANSPARANTIE")
print("-" * 40)
if transparantie_rapporten:
    velden = list(transparantie_rapporten[0].keys())
    print(f"  Elk rapport bevat {len(velden)} metadata-velden:")
    for v in velden:
        print(f"    - {v}")
    print(f"   Transparantie werkt — {len(transparantie_rapporten)} rapporten gegenereerd")
else:
    print("   Geen transparantierapporten gevonden")

# Test 3: Logging
print("\n3. LOGGING")
print("-" * 40)
alle_logs_aanwezig = all(os.path.exists(p) for p in logbestanden.values())
if alle_logs_aanwezig:
    print(f"  Logging werkt — alle {len(logbestanden)} logbestanden aanwezig")
else:
    print("   Niet alle logbestanden gevonden")

# Conclusie
print(f"""
{'='*60}
CONCLUSIE DV5 — Guardrails
{'='*60}

Drie guardrails zijn geïmplementeerd op POC-niveau:

1. ROLGEBONDEN TOEGANG
   Studenten zien alleen formatieve feedback. Docenten zien
   daarnaast modelinstellingen, prompts en logs. Geïmplementeerd
   via een Python dictionary met rechten per rol.

2. TRANSPARANTIE
   Bij elke feedback-output wordt een transparantierapport
   gegenereerd met model, temperatuur, promptvariant, timestamp
   en verwerkingsdetails. Dit maakt de output reproduceerbaar.

3. LOGGING
   Alle Ollama-aanroepen worden gelogd in JSONL-formaat met
   prompt, response en metadata. Afbeeldingen die niet verwerkt
   konden worden zijn apart gelogd. Alle geanonimiseerde teksten
   en resultaten zijn opgeslagen als JSON.

AANSLUITING OP PRINCIPES:
- Privacy: anonimisering verwijdert namen en studentnummers
  voordat tekst naar de LLM gaat. Alles draait lokaal.
- Reproduceerbaarheid: transparantierapporten bevatten alle
  parameters om een aanroep exact te herhalen.
- Controleerbaarheid: logging maakt het mogelijk om achteraf
  te verifiëren wat het model heeft ontvangen en gegenereerd.
""")

TEST GUARDRAILS

1. ROLGEBONDEN TOEGANG
----------------------------------------
  Student ziet: ziet_feedback
  Docent ziet:  ziet_feedback, ziet_modelinstellingen, ziet_prompt, ziet_logs, ziet_scores
   Rolgebonden toegang werkt — student ziet 1 velden, docent ziet 5 velden

2. TRANSPARANTIE
----------------------------------------
  Elk rapport bevat 15 metadata-velden:
    - groep
    - criterium
    - prompt_variant
    - model
    - temperatuur
    - max_tokens
    - prompt_lengte_tekens
    - response_lengte_tekens
    - verwerkingstijd_seconden
    - timestamp
    - input_tekst_max_tekens
    - ollama_url
    - anonimisering_toegepast
    - afbeeldingen_verwerkt
    - afbeeldingen_gelogd
   Transparantie werkt — 6 rapporten gegenereerd

3. LOGGING
----------------------------------------
  Logging werkt — alle 3 logbestanden aanwezig

CONCLUSIE DV5 — Guardrails

Drie guardrails zijn geïmplementeerd op POC-niveau:

1. ROLGEBONDEN TOEGANG
   Studenten zien alleen formatieve feedb